In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import yaml
import sys
import dask
import socket
import calendar
import pandas as pd
import os

import importlib
import utils
importlib.reload(utils)

# import utilities
from rossby_utils import cds_sort_utils as cds_sort

import warnings
warnings.filterwarnings('ignore')
import gc

### Set up dask cluster.  Note, you will need to change
- local_directory
- project

In [2]:
from dask_jobqueue import PBSCluster
from dask.distributed import Client
dask.config.set({"distributed.scheduler.worker_saturation":1.0})
dask.config.set({"optimization.fuse.active": False})
dask.config.set({
    "distributed.worker.memory.target": 0.6,
    "distributed.worker.memory.spill": 0.7,
    "distributed.worker.memory.pause": 0.8,
    "distributed.worker.memory.terminate": 0.95,
})

cluster = PBSCluster(
    cores = 1,
    memory = '30GB',
    processes = 1,
    queue = 'casper',
    local_directory = '/glade/derecho/scratch/islas/dask_tmp/',
    resource_spec = 'select=1:ncpus=1:mem=30GB',
    project='P04010022',
    walltime='01:00:00',
    interface='mgt')

# scale up
cluster.scale(12)
#cluster.adapt(minimum=1, maximum=12)

# change your urls to the dask dashboard so that you can see it
hostname = socket.getfqdn()
dask.config.set({"distributed.dashboard.link":
        f"https://ondemand.hpc.ucar.edu/rnode/{hostname}/{{port}}/status"
})

client = Client(cluster)

#### Wait after the next cell until you get the workers

In [3]:
cluster

Dashboard: https://ondemand.hpc.ucar.edu/rnode/crhtc67.hpc.ucar.edu/37037/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.18.206.68:40031,Workers: 0
Dashboard: https://ondemand.hpc.ucar.edu/rnode/crhtc67.hpc.ucar.edu/37037/status,Total threads: 0
Started: Just now,Total memory: 0 B


#### Read in system information.  Each model has multiple systems that are used for different years in the later part of the record.  The climatology from the corresponding system will be used to remove the lead dependent climatology and the newest system will be used for the climatological period

In [7]:
with open("../../../DATA_DOWNLOAD/system_year_info.yaml") as f:
    model_info = yaml.safe_load(f)

#### Set up file info

In [8]:
# directory containing CDS downloads
basepath = "/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/"
# path to store the output
outpath="/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/"
# initialization month
init_mon = 11
# forecast months (NDJFM)
fc_mon = [11,12,1,2,3] 
# variable to sort out (our naming convention)
var='slp'
# the variable name in the raw downloaded files
data_var='msl'

#### Loop over models to sort

In [9]:
models=['ECMWF']

In [16]:
dask.config.set({"array.chunk-size":"256MiB"})
chunks={"number": 28,
        "time": 1,
        "step": 1,
        "latitude": 180,
        "longitude": 360
       }


for imodel in models:
    info = model_info[imodel]
    systems = info['forecast'].keys()

    #!!!
    #systems = list(systems)
    #systems = systems[len(systems)-1:len(systems)]
    for isystem in systems:
        print('system=',isystem)
        
        datdir=basepath+imodel+'/grib/'+var+'/'+str(isystem)+'/'
        outdir=outpath+imodel+'/nc/'+var+'/'+str(isystem)+'/'
        os.makedirs(outdir, exist_ok=True)
        
        # loop over forecast months
        alldat_pc_ALLM=[]
        alldat_hc_ALLM=[]
        alldat_fc_ALLM=[]
        for imon in fc_mon:
            
            print('month=',imon)
            # hindcast and forecast data and precast (years before hindcast)
            dat_pc= xr.open_mfdataset(datdir+'precast_'+var+'_init'+str(init_mon).zfill(2)+'_mon'+str(imon).zfill(2)+'.grib', chunks = chunks)
            dat_hc = xr.open_mfdataset(datdir+'hindcast_'+var+'_init'+str(init_mon).zfill(2)+'_mon'+str(imon).zfill(2)+'.grib', chunks=chunks)
            dat_fc = xr.open_mfdataset(datdir+'forecast_'+var+'_init'+str(init_mon).zfill(2)+'_mon'+str(imon).zfill(2)+'.grib', chunks=chunks)

            # add time axis in the case there's only one forecast
            if "time" not in dat_fc.dims:
                dat_fc = dat_fc.expand_dims(time=[dat_fc.valid_time.values])
            
            # setting valid_time to be 1st of the following month to deal with valid_time inconsistencies across initializations
            nextmon = imon + 1
            if nextmon > 12:
                nextmon = 1
            year_pc = dat_pc.valid_time.dt.year
            day_1_pc = (year_pc.astype(str)+'-'+str(nextmon).zfill(2)+'-01')
            year_hc = dat_hc.valid_time.dt.year
            day_1_hc = (year_hc.astype(str)+'-'+str(nextmon).zfill(2)+'-01')
            year_fc = dat_fc.valid_time.dt.year
            day_1_fc = (year_fc.astype(str)+'-'+str(nextmon).zfill(2)+'-01')

            dat_pc['valid_time'] = day_1_pc.astype("datetime64[ns]")
            dat_hc['valid_time'] = day_1_hc.astype("datetime64[ns]")
            dat_fc['valid_time'] = day_1_fc.astype("datetime64[ns]")
            
            # using sort_predictions function in utils
            dat_monthly_pc = cds_sort.sort_predictions(dat_pc, data_var)
            dat_monthly_hc = cds_sort.sort_predictions(dat_hc, data_var)
            dat_monthly_fc = cds_sort.sort_predictions(dat_fc, data_var)       

            # do some renaming of dimensions and transpose
            dat_monthly_pc = dat_monthly_pc.rename({data_var:var, 'latitude':'lat', 'longitude':'lon'})
            dat_monthly_pc = dat_monthly_pc.transpose('member','time','lat','lon')
            
            dat_monthly_hc = dat_monthly_hc.rename({data_var:var, 'latitude':'lat', 'longitude':'lon'})
            dat_monthly_hc = dat_monthly_hc.transpose('member','time','lat','lon')

            dat_monthly_fc = dat_monthly_fc.rename({data_var:var, 'latitude':'lat', 'longitude':'lon'})
            dat_monthly_fc = dat_monthly_fc.transpose('member','time','lat','lon')

            # Give it a dimension that corrresponds to the initialization year, instead of time
            if imon == fc_mon[0]: # note this won't necessarily work if you decide not to output the first month
                years_pc = dat_monthly_pc.time.dt.year
                years_hc = dat_monthly_hc.time.dt.year
                years_fc = dat_monthly_fc.time.dt.year
            dat_monthly_pc['time'] = years_pc
            dat_monthly_pc = dat_monthly_pc.rename(time='year')   
            dat_monthly_hc['time'] = years_hc
            dat_monthly_hc = dat_monthly_hc.rename(time='year')
            dat_monthly_fc['time'] = years_fc
            dat_monthly_fc = dat_monthly_fc.rename(time='year')

            if "step" in dat_monthly_pc:
                dat_monthly_pc = dat_monthly_pc.drop_vars("step")
            if "step" in dat_monthly_hc:
                dat_monthly_hc = dat_monthly_hc.drop_vars("step")
            if "step" in dat_monthly_fc:
                dat_monthly_fc = dat_monthly_fc.drop_vars("step")

            dat_monthly_pc_anoms_ALLM = dat_monthly_pc[var] - dat_monthly_pc[var].mean(['member','year'])
            dat_monthly_hc_anoms_ALLM = dat_monthly_hc[var] - dat_monthly_hc[var].mean(['member','year'])
            dat_monthly_fc_anoms_ALLM = dat_monthly_fc[var] - dat_monthly_hc[var].mean(['member','year'])

            if 'member' in dat_monthly_pc.valid_time.dims:
                times_pc = dat_monthly_pc.valid_time.isel(member=0).rename('time')
            else:
                times_pc = dat_monthly_pc.valid_time.rename('time')
            if 'member' in dat_monthly_hc.valid_time.dims:
                times_hc = dat_monthly_hc.valid_time.isel(member=0).rename('time')
            else:
                times_hc = dat_monthly_hc.valid_time.rename('time')
            if 'member' in dat_monthly_fc.valid_time.dims:
                times_fc = dat_monthly_fc.valid_time.isel(member=0).rename('time')
            else:
                times_fc = dat_monthly_fc.valid_time.rename('time')
            
            datout_pc_ALLM = xr.merge([dat_monthly_pc_anoms_ALLM, times_pc])
            datout_hc_ALLM = xr.merge([dat_monthly_hc_anoms_ALLM, times_hc])
            datout_fc_ALLM = xr.merge([dat_monthly_fc_anoms_ALLM, times_fc])

            alldat_pc_ALLM.append(datout_pc_ALLM)
            alldat_hc_ALLM.append(datout_hc_ALLM)
            alldat_fc_ALLM.append(datout_fc_ALLM)


        alldat_pc_ALLM = xr.concat(alldat_pc_ALLM, dim='lead')
        alldat_pc_ALLM['lead'] = np.arange(1,len(fc_mon)+1,1)
        alldat_pc_ALLM = alldat_pc_ALLM.transpose('member','lead','year','lat','lon')

        alldat_hc_ALLM = xr.concat(alldat_hc_ALLM, dim='lead')
        alldat_hc_ALLM['lead'] = np.arange(1,len(fc_mon)+1,1)
        alldat_hc_ALLM = alldat_hc_ALLM.transpose('member','lead','year','lat','lon')

        alldat_fc_ALLM = xr.concat(alldat_fc_ALLM, dim='lead')
        alldat_fc_ALLM['lead'] = np.arange(1,len(fc_mon)+1,1)
        alldat_fc_ALLM = alldat_fc_ALLM.transpose('member','lead','year','lat','lon')

        alldat_pc_ALLM.to_netcdf(outdir+'precast_anoms_ALLM_'+var+'.nc')
        alldat_hc_ALLM.to_netcdf(outdir+'hindcast_anoms_ALLM_'+var+'.nc')
        alldat_fc_ALLM.to_netcdf(outdir+'forecast_anoms_ALLM_'+var+'.nc')
        
        del alldat_hc_ALLM, alldat_fc_ALLM
        del dat_hc, dat_fc
        del dat_monthly_hc, dat_monthly_fc
        del dat_monthly_hc_anoms_ALLM, dat_monthly_fc_anoms_ALLM
        del datout_hc_ALLM, datout_fc_ALLM
        del times_hc, times_fc
        gc.collect()

system= 51
month= 11
month= 12
month= 1
month= 2
month= 3


NameError: name 'dat_monthly_hc_anoms_INDM' is not defined

In [17]:
cluster.close()